# NSE data inspection

Load market data into pandas DataFrames for ad-hoc exploration. Analysis cells will be added as needed.

- **Stocks:** all `nse_cm_*.parquet` files under `PARQUET_DATA_DIR` → `df`
- **NIFTY 50 index:** `NIFTY50.parquet` → `nifty_df` (when present)

Override data root with env `QUANT_DATA_DIR` (see `quant.config`).

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "quant").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

for pkg in ("pandas", "pyarrow", "duckdb"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import duckdb
import pandas as pd

from quant.config import DATA_DIR, PARQUET_DATA_DIR, NIFTY50_PATH

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("DATA_DIR:", DATA_DIR)
print("PARQUET_DATA_DIR:", PARQUET_DATA_DIR, "exists:", PARQUET_DATA_DIR.exists())
print("NIFTY50_PATH:", NIFTY50_PATH, "exists:", NIFTY50_PATH.exists())

In [ ]:
def stock_parquet_files(data_dir: Path) -> list[Path]:
    return sorted(
        p
        for p in data_dir.rglob("nse_cm_*.parquet")
        if "results" not in p.parts and "indices" not in p.parts
    )


def load_all_stock_bars(files: list[Path], *, data_dir: Path) -> pd.DataFrame:
    """Load every daily stock parquet into one DataFrame (DuckDB union)."""
    if not files:
        raise FileNotFoundError(f"No nse_cm_*.parquet under {data_dir}")
    con = duckdb.connect()
    paths_sql = ", ".join(f"'{str(p).replace(chr(39), chr(39) * 2)}'" for p in files)
    sql = f"""
    SELECT *
    FROM read_parquet([{paths_sql}], union_by_name=true)
    """
    out = con.execute(sql).df()
    cols = {c: c.lower().strip() for c in out.columns}
    out = out.rename(columns=cols)
    date_col = next((c for c in ("date", "datetime", "timestamp") if c in out.columns), None)
    if date_col:
        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    sym_col = next((c for c in ("symbol", "ticker", "tradingsymbol") if c in out.columns), None)
    if sym_col and sym_col != "symbol":
        out = out.rename(columns={sym_col: "symbol"})
    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper().str.strip()
    return out


files = stock_parquet_files(PARQUET_DATA_DIR)
print(f"Found {len(files)} stock parquet files")
if files:
    print("First:", files[0].name, "| Last:", files[-1].name)

In [ ]:
# Full stock universe (may be large — ensure sufficient RAM)
df = load_all_stock_bars(files, data_dir=PARQUET_DATA_DIR)
print("df shape:", df.shape)
print("Columns:", list(df.columns))
date_col = "date" if "date" in df.columns else None
if date_col:
    print("Date range:", df[date_col].min(), "→", df[date_col].max())
if "symbol" in df.columns:
    print("Symbols:", df["symbol"].nunique())
df.head()

In [ ]:
nifty_df = None
if NIFTY50_PATH.exists():
    nifty_df = pd.read_parquet(NIFTY50_PATH)
    nifty_df.columns = [c.lower().strip() for c in nifty_df.columns]
    if "date" in nifty_df.columns:
        nifty_df["date"] = pd.to_datetime(nifty_df["date"], errors="coerce")
    print("nifty_df shape:", nifty_df.shape)
    display(nifty_df.head())
else:
    print("NIFTY50 parquet not found — skip index series.")

## Optional: pre-built explosive-moves panel

If you already ran `explosive_moves_research.ipynb` Phase 2, you can load the stitched panel instead of re-reading every source file:

In [ ]:
PANEL_PATH = DATA_DIR / "results" / "explosive_moves" / "panel" / "ohlcv_panel.parquet"
panel_df = None
if PANEL_PATH.exists():
    panel_df = pd.read_parquet(PANEL_PATH)
    panel_df["Date"] = pd.to_datetime(panel_df["Date"])
    print("panel_df:", panel_df.shape, panel_df["Date"].min(), "→", panel_df["Date"].max())
    display(panel_df.head())
else:
    print("No stitched panel at", PANEL_PATH)

## Analysis (placeholder)

Add inspection / QA cells below.